# IEEE-CIS SCARF Benchmark on Colab

Notebook nay dung de benchmark nhanh `SCARF` standalone tren Google Colab truoc khi xem xet fusion vao he thong chinh.

## Runtime

Trong Colab, dat `Runtime -> Change runtime type -> Hardware accelerator -> GPU` truoc khi chay.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_REPO = 'https://github.com/Thanh-000/FDB1-DoAn.git'
REPO_PARENT = '/content/drive/MyDrive'
REPO_NAME = 'FDB1-DoAn'
REPO_DIR = f'{REPO_PARENT}/{REPO_NAME}'

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip'
EXTRACT_DIR = '/content/ieee-fraud-detection'

FOLD_INDEX = 0
N_SPLITS = 5
PRETRAIN_EPOCHS = 10
DOWNSTREAM_EPOCHS = 10
BATCH_SIZE = 2048
HIDDEN_DIM = 256
EMB_DIM = 128
PROJ_DIM = 128
LR = 1e-3
CORRUPTION_RATE = 0.6
FREEZE_ENCODER = False
NO_PRETRAIN = False


In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('REPO_DIR =', REPO_DIR)


In [ ]:
import os
import zipfile
import glob

if ZIP_PATH and not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)

txn_files = glob.glob(f'{EXTRACT_DIR}/**/train_transaction.csv', recursive=True)
id_files = glob.glob(f'{EXTRACT_DIR}/**/train_identity.csv', recursive=True)

print('txn:', txn_files[:1])
print('id :', id_files[:1])

if not txn_files or not id_files:
    raise FileNotFoundError('Could not locate train_transaction.csv/train_identity.csv after extraction')

DATA_DIR = os.path.dirname(txn_files[0])
print('DATA_DIR =', DATA_DIR)


In [ ]:
%cd "$REPO_DIR"
!pwd
!ls
!ls scarf


In [ ]:
import sys
import torch
import sklearn
import pandas as pd

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('sklearn:', sklearn.__version__)
print('pandas:', pd.__version__)


In [ ]:
cmd = [
    'python', 'run_ieee_scarf.py',
    '--data-dir', DATA_DIR,
    '--fold-index', str(FOLD_INDEX),
    '--n-splits', str(N_SPLITS),
    '--pretrain-epochs', str(PRETRAIN_EPOCHS),
    '--downstream-epochs', str(DOWNSTREAM_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--emb-dim', str(EMB_DIM),
    '--proj-dim', str(PROJ_DIM),
    '--lr', str(LR),
    '--corruption-rate', str(CORRUPTION_RATE),
]
if FREEZE_ENCODER:
    cmd.append('--freeze-encoder')
if NO_PRETRAIN:
    cmd.append('--no-pretrain')

print('Running:')
print(' '.join(cmd))


In [ ]:
import subprocess

subprocess.run(cmd, check=True)
